# MP19 — CNN Image Reconstruction

A reconstruction network maps an image back to an image of the same shape. This is a useful mechanics check before super-resolution. A constant-output baseline predicts the training-set channel mean everywhere. It is weak but exposes whether a network learned spatial structure.

## Mathematics, shapes and worked example

For N examples, C channels and H×W pixels, MSE = sum((prediction-target)²)/(NCHW). A convolutional encoder extracts features and a reconstruction head maps features back to C channels. Here two padded 3×3 convolutions surround ReLU, with width eight. For prediction [0.2,0.8] and target [0,1], MSE=(0.04+0.04)/2=0.04. Shapes stay N×3×H×W; zero padding can introduce border errors.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
torch.manual_seed(19)
from rssr.cnn_labs import Reconstruction
x=torch.rand(2,3,8,9)
model=Reconstruction()
y=model(x)
assert y.shape==x.shape
loss=(y-x).square().mean();loss.backward()
assert all(p.grad is not None for p in model.parameters())
print('Input/output',tuple(x.shape),tuple(y.shape))

## Reproducible experiment

Run `python scripts/cnn_lessons_experiment.py` from the repository root. The script first checks a small synthetic input/output shape. It then uses four training and four validation NAIP scenes, split before tiling: 16 nonoverlapping 32×32 RGB patches per scene. HR stored DN are divided by 10000, not claimed as reflectance. SR inputs use Gaussian sigma 0.8 HR pixels and area x2 downsampling. Identity reconstruction uses HR as both input and target.

Training: CPU float32, two threads, seed 19, Adam learning rate 0.001, batch four, 300 updates, MSE, no augmentation or scheduler. Evaluate the final fixed step; no validation-based checkpoint search. Teaching weights are regenerated and not committed. The constant baseline uses training-channel means. SR baselines use bicubic with align_corners=False. Metrics use range one, border two, all RGB channels; equal patches per scene give equal scene weighting.

Data: existing checksummed NAIP/OpenSR crops, public domain with USDA/OpenSR attribution. No new imagery download. Final research weights/protocol/test results are untouched by these separate teaching exercises. No learner mastery claim.

## Observed result and interpretation

Identity reconstruction reached 36.144 dB versus 27.174 dB for the training-mean constant baseline on validation. It still loses detail and shows boundary errors. This demonstrates learnability, not useful super-resolution: input and target are the same image.

In [ ]:
report=json.loads((ROOT/'reports/mp19-mp22-cnn.json').read_text())
selected=['identity']
for row in report['results']:
 if row['model'] in selected:
  print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import Image,display
display(Image(filename=str(ROOT/'reports/mp19-mp22-cnn.png')))

## Practice — solve first

1. Why compute the constant baseline from training images only?
2. What changes if every predicted pixel equals the target?
3. Why can a padded convolution make border errors on an identity task?

## Worked solutions

1. Using validation values would leak evaluation information into the baseline.
2. MSE is zero; PSNR is positive infinity for a positive declared range.
3. The boundary sees artificial zeros rather than the same neighborhood statistics as interior pixels.

## Limitations and next question

One seed and a short budget describe an engineering demonstration. Interpolation, padding, input distribution and unequal parameter counts affect results. Investigate these factors on validation before any new frozen research test. Generated answers do not assess learner mastery.

[Authoritative brief](../stage_05_cnn_sr/MP19_cnn_image_reconstruction/README.md) · [Local/Colab setup](../README.md)